# FinAlign: development selection and paired evaluation

Select query-expansion and lexical-fusion weights on **500 dev queries**, freeze the choices,
then evaluate five systems on **648 test queries**. The report contains nDCG@10, Recall@10,
paired differences, 20,000 bootstrap samples and three two-sided randomization tests with Holm correction.

1. Select **Runtime → Change runtime type → T4 GPU**.
2. Select **Runtime → Run all**, then authorize the Google Drive mount.
3. Keep the default run name to resume the same experiment after an interruption.
4. Download the results ZIP when the final cell finishes. Checkpoints also remain in Drive.

The notebook reuses the completed passage index. It does not rebuild embeddings, edit GitHub,
upload to Hugging Face or alter the live demo. No token or paid inference service is needed.
Models and the index use temporary Colab disk; checkpoints use your Google Drive storage.

The original project used the test set during development. This follow-up retains that limitation:
it is **not an independent, previously untouched holdout**. See the
[published plan](https://github.com/pdmdp/FinAlign/blob/main/demo/EXPERIMENTS.md).

Free Colab GPU access and session lengths vary; see the [Colab FAQ](https://research.google.com/colaboratory/faq.html).
This is a full query evaluation, not model training. Its duration depends on GPU availability and generation speed;
the notebook prints progress throughout. Run only one copy for a given Drive folder.


## 1. Mount Drive and choose the run folder

The default folder is `MyDrive/FinAlign/experiments/followup-v1`.
Keep this name when reconnecting. Use a new name only for a deliberately new experiment;
do not delete the frozen configuration to retune an experiment after test results have been seen.


In [ ]:
import os
import re
import sys
import json
import hashlib
import shutil
import subprocess
from pathlib import Path
from google.colab import drive

RUN_NAME = "followup-v1"
if not re.fullmatch(r"[a-zA-Z0-9_-]{1,60}", RUN_NAME):
    raise ValueError("Use letters, digits, underscores or hyphens for RUN_NAME.")
drive.mount("/content/drive")
ROOT = Path("/content/finalign-followup")
SOURCE = ROOT / "source"
RUN = Path("/content/drive/MyDrive/FinAlign/experiments") / RUN_NAME
ROOT.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)
print("Checkpoints:", RUN)
print("Existing freeze:", (RUN / "frozen_config.json").exists())


## 2. Download the fixed benchmark source

This notebook pins a GitHub commit and checks every downloaded file. It uses read-only imports
from the demo's retrieval modules. A later change to the public demo cannot change this run.


In [ ]:
from urllib.request import urlopen

SOURCE_REVISION = "f0c006a328694de8c31bb09b398a19136f027032"
SOURCE_FILES = {
    "demo/finalign/__init__.py": "e10f80ad2147c5156ed0788b5284ce8685403a9a8b7f897f6500d086f8130ca4",
    "demo/finalign/core.py": "85da32491835d00c722de08e1ef5c9d5ea40a3d79d0e048043f7755614aaf5f8",
    "demo/finalign/index.py": "88a2412faddcb34f739dd97dddbbaf629e62565277d1b0b8bd59f8884ee4dde8",
    "demo/finalign/neural.py": "4a79bf33b892e4d4c0b91ee7221a4998ff4ea4fd1689ffe5450fa57b3f6c3dc3",
    "demo/models.lock.json": "d3dbdbbd4ed418014231eb595dd913c87823511fde82012dd4e9714c9e8232e4",
    "demo/EXPERIMENTS.md": "7b176062efd07bb33f4416b84c5b4ce961a234388eea8605b63a51c47f4f4075",
    "experiments/requirements.txt": "18fd5ed83aa82a215d67e06db42a885acc917468514b2661e350f50c292b956d",
    "experiments/followup/__init__.py": "c8cda6904ca8fdcdead168de7b5cc2c50411c2e579496304ef5d7d55b496b434",
    "experiments/followup/protocol.py": "58ef23158171209c0a078423471ce55115e4edfc52f6614416b5b8b6f2fd6291",
    "experiments/followup/storage.py": "88ae9b3927bc3301e6f4be632528bf12704c3844d920226d22bdba055a40e296",
    "experiments/followup/backend.py": "809f428f2935f0de9aa2b81b5a459458badd4ce4109c16d14e781f4709029333",
    "experiments/followup/runner.py": "742c7b6e58a27e1449079bf5636f1dcf8445c60113b0500135474d4285ee6a37",
    "experiments/followup/report.py": "b198de89eb4d4a951ebe18a40c9a8f9f407f3ca9eab10d7f44e53a4a82efe0d5",
    "experiments/tests/test_followup.py": "8dc314ea9e3aa0dc652b79df57cb624e1609d1f93136a8bce2a6aca07da2bdcf"
}

def sha256(path):
    value = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            value.update(block)
    return value.hexdigest()

def fetch_checked(url, destination, expected):
    destination = Path(destination)
    if destination.exists() and sha256(destination) == expected:
        return
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_name(destination.name + ".part")
    with urlopen(url, timeout=120) as response, partial.open("wb") as stream:
        shutil.copyfileobj(response, stream, length=1024 * 1024)
    if sha256(partial) != expected:
        raise ValueError(f"Checksum mismatch: {destination.name}. Retry this cell.")
    partial.replace(destination)

if not re.fullmatch(r"[0-9a-f]{40}", SOURCE_REVISION):
    raise ValueError("This notebook needs its published source commit.")
for name, expected in SOURCE_FILES.items():
    fetch_checked(f"https://raw.githubusercontent.com/pdmdp/FinAlign/{SOURCE_REVISION}/{name}",
                  SOURCE / name, expected)
print("Verified", len(SOURCE_FILES), "source files at", SOURCE_REVISION)


## 3. Install the isolated runtime

This takes a few minutes on a new Colab runtime. The benchmark gets its own Python environment,
CUDA PyTorch and Java JDK. When resuming, the saved dependency lock restores the original package versions.


In [ ]:
if not shutil.which("nvidia-smi"):
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU and reconnect.")
print(subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True))
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
java = next(p for p in Path("/usr/lib/jvm").glob("java-17-openjdk-*") if (p/"bin/javac").exists())
VENV = ROOT / "venv"
PYTHON = VENV / "bin/python"
if not PYTHON.exists():
    bootstrap = ROOT / "bootstrap"
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--target", str(bootstrap),
                    "virtualenv==20.35.3"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", "--no-download", str(VENV)],
                   env={**os.environ, "PYTHONPATH": str(bootstrap)}, check=True)
RUN_ENV = {**os.environ, "JAVA_HOME": str(java), "PATH": str(java/"bin") + os.pathsep + os.environ["PATH"],
           "PYTHONPATH": str(SOURCE/"experiments") + os.pathsep + str(SOURCE/"demo"),
           "HF_HOME": str(ROOT/"model-cache"), "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1",
           "TOKENIZERS_PARALLELISM": "false", "CUBLAS_WORKSPACE_CONFIG": ":4096:8",
           "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2"}
subprocess.run([str(PYTHON), "-m", "pip", "install", "--quiet", "--no-cache-dir", "torch==2.8.0",
                "--index-url", "https://download.pytorch.org/whl/cu126"], env=RUN_ENV, check=True)
lock = RUN / "environment-lock.txt"
requirements = lock if lock.exists() else SOURCE / "experiments/requirements.txt"
subprocess.run([str(PYTHON), "-m", "pip", "install", "--quiet", "--no-cache-dir", "-r", str(requirements)],
               env=RUN_ENV, check=True)
subprocess.run([str(PYTHON), "-m", "pip", "check"], env=RUN_ENV, check=True)
subprocess.run([str(PYTHON), "-c", "import torch; assert torch.cuda.is_available(); print('CUDA ready:', torch.cuda.get_device_name(0))"],
               env=RUN_ENV, check=True)
print("Environment ready.")


## 4. Reuse the validated index

Download the existing 218 MB archive from a fixed Space revision. Every index file is verified.
There is no upload and no use of the Space's ZeroGPU inference quota.


In [ ]:
SPACE_REVISION = "99436a31eaf7f9d9a2d605adf07cf95e6b5ad1b6"
INDEX_SHA256 = "45b9da96081deb1f2f8c946125c413bafe7bad464fcc0b3c9ccec21300036366"
ARCHIVE = ROOT / "fiqa-index.zip"
INDEX = ROOT / "index"
print("Downloading/checking the index...")
fetch_checked(f"https://huggingface.co/spaces/pdmdp/finalign/resolve/{SPACE_REVISION}/data/fiqa-index.zip?download=true",
              ARCHIVE, INDEX_SHA256)
subprocess.run([str(PYTHON), "-c",
    "import sys; from finalign.index import Index,unpack; "
    "i=Index(unpack(sys.argv[1],sys.argv[2])); "
    "assert i.ready; print(len(i.documents),'documents,',len(i.passage_ids),'passages verified')",
    str(ARCHIVE), str(INDEX)], env=RUN_ENV, check=True)

def run_stage(stage):
    from collections import deque
    from datetime import datetime, timezone
    command = [str(PYTHON), "-u", "-m", "followup.runner", stage,
               "--index", str(INDEX), "--run", str(RUN), "--device", "cuda",
               "--source-revision", SOURCE_REVISION]
    log_path = RUN / "logs" / f"{stage}.log"
    log_path.parent.mkdir(parents=True, exist_ok=True)
    tail = deque(maxlen=20)
    with log_path.open("a", encoding="utf-8") as log:
        log.write(f"\n{datetime.now(timezone.utc).isoformat()} stage={stage}\n")
        log.flush()
        with subprocess.Popen(command, cwd=SOURCE, env=RUN_ENV,
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1) as process:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
                tail.append(line)
            returncode = process.wait()
    if returncode:
        raise RuntimeError(f"{stage} failed (exit {returncode}). Full log: {log_path}. "
                           "Completed query checkpoints are retained. "
                           "Section 9 can download them without rerunning the experiment.\n\n"
                           + "".join(tail))


## 5. Check the pipeline on dev queries

First run the automated checks, then two real dev queries through all stages.
The first run downloads BGE, Qwen and monoT5; this is slower than subsequent queries.
The component cache is reused during the full dev run. These two queries do not replace the 500-query selection.


In [ ]:
checks = subprocess.run([str(PYTHON), "-m", "unittest", "discover", "-s", str(SOURCE/"experiments/tests"), "-v"],
                        cwd=SOURCE, env=RUN_ENV, text=True, capture_output=True)
if checks.returncode:
    print(checks.stdout)
    print(checks.stderr)
    checks.check_returncode()
print(checks.stderr.strip().splitlines()[-3:])
run_stage("smoke")


## 6. Select parameters on dev and freeze

Five expansion settings are compared on all 500 dev queries. After selecting one, five BM25 weights
are compared using the same cached inputs. The tie rules prefer the historical expansion setting and
the smallest lexical weight. This cell writes `dev_selection.json`, `frozen_config.json` and its SHA-256
**before any test query is evaluated**. On resume it reuses an existing valid freeze.


In [ ]:
run_stage("dev")
frozen = json.loads((RUN / "frozen_config.json").read_text())
print("Selected expansion weights:", frozen["weights"])
print("Selected BM25 alpha:", frozen["alpha"])
print("Frozen SHA-256:", (RUN / "frozen_config.sha256").read_text().strip())


## 7. Evaluate all 648 test queries

Five rankings are retained for every query: BM25 (`L`), original BGE (`B`), expanded BGE (`E`),
lexical fusion (`H`) and monoT5 (`F`). Completed queries are saved individually.
If interrupted, reconnect and run the notebook with the same run name: valid checkpoints are reused.
A runtime failure stops the stage; it is never converted into a zero effectiveness score.


In [ ]:
run_stage("test")


## 8. Analyze the saved results

The report is produced only after the complete paired set is available. It includes all three
planned comparisons (`E−B`, `H−E`, `F−H`), even if a difference is negative or exactly zero.
Bootstrap intervals are marginal 95% intervals; planned significance decisions use Holm-adjusted p-values.

This stage reads the saved rankings; it does not repeat model inference. If it fails,
the actual error is printed and saved in `logs/report.log`. Run section 9 on its own to
download the checkpoints and log before troubleshooting. Do not delete or alter the freeze.


In [ ]:
run_stage("report")
from IPython.display import display, Markdown, SVG
report_text = (RUN / "report/REPORT.md").read_text()
display(Markdown(report_text.replace("![Per-query nDCG differences](paired-differences.svg)", "")))
display(SVG(filename=str(RUN / "report/paired-differences.svg")))


## 9. Download the saved run

This cell only packages existing files. You can run it separately even if the report cell failed;
no queries are recomputed. An archive downloaded after an error may be incomplete, so retain
the logs and check completion before interpreting its results.

The ZIP includes the frozen configuration, dependency lock, query variants, component caches,
rankings, per-query metrics, report files and logs available so far. It contains no model weights.
Keep the folder on Drive to resume or audit the run. Do not change the parameter grid after
inspecting test outputs.


In [ ]:
import zipfile
from google.colab import files
OUTPUT = ROOT / f"finalign-{RUN_NAME}-results.zip"
with zipfile.ZipFile(OUTPUT, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as bundle:
    for path in sorted(RUN.rglob("*")):
        if path.is_file() and path.name != ".run.lock" and not path.name.endswith(".tmp"):
            bundle.write(path, Path(RUN_NAME) / path.relative_to(RUN))
print("Results:", RUN / "report")
print("Download:", OUTPUT.name, f"({OUTPUT.stat().st_size / 1_000_000:.1f} MB)")
files.download(str(OUTPUT))


Code: [FinAlign](https://github.com/pdmdp/FinAlign), MIT. Data and derived index:
[FiQA / BEIR](https://huggingface.co/datasets/BeIR/fiqa), CC BY-SA 4.0.
Pretrained models retain their own licenses.

Original project: Matteo Pietro Di Pilato, Lorenzo Cinquemani and Lorenzo Marchioni.
This notebook implements the subsequent evaluation plan; it leaves the submitted notebook and live demo unchanged.
